# Amazon Bedrock boto3 Prerequisites

> *This notebook should work well with the **`Data Science 3.0`** kernel in SageMaker Studio*

---

In this demo notebook, we demonstrate how to use the [`boto3` Python SDK](https://boto3.amazonaws.com/v1/documentation/api/latest/index.html) to work with [Amazon Bedrock](https://aws.amazon.com/bedrock/) Foundation Models.

---

> **Updated November 2026.** Amazon Titan Text, Anthropic Claude 3.x and Stable Diffusion XL reached end of life on Amazon Bedrock (see [Model lifecycle](https://docs.aws.amazon.com/bedrock/latest/userguide/model-lifecycle.html)). This notebook now uses [Amazon Nova 2 Lite](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-amazon-nova-2-lite.html) through the [Converse API](https://docs.aws.amazon.com/bedrock/latest/userguide/conversation-inference-call.html) and [Claude Haiku 5.5](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-anthropic-claude-haiku-5-5.html) / [Claude Sonnet 5.5](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-anthropic-claude-sonnet-5-5.html) through cross-Region inference profiles. Claude 4.7+ no longer accepts `temperature`, `top_p` or `top_k` ([Anthropic docs](https://platform.claude.com/docs/en/models/haiku-5-5/overview)), so those parameters were removed from the Claude requests.

<div class="alert alert-block alert-info">
⚠️ <b>Before you get started with this workshop...</b> make sure you have access you have manually enabled access to the following models in the "Model access" tab:

<ul>
  <li>Amazon Nova 2 Lite</li>
  <li>Anthropic Claude Haiku 5.5</li>
  <li>Amazon Titan Embeddings G1 - Text</li>
  <li>Stability AI Stable Diffusion 3.5 Large (available in us-west-2)</li>
</ul>  

</div>

Remember that you can modify model access at any point of time, but if you are running this workshop at an AWS event, it is likely you have limited access to third party models.


## Prerequisites

Run the cells in this section to install the packages needed by the notebooks in this workshop. ⚠️ You will see pip dependency errors, you can safely ignore these errors. ⚠️

IGNORE ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.

In [1]:
%pip install --upgrade "boto3>=1.44" "botocore>=1.44"


---

## Create the boto3 client

Interaction with the Bedrock API is done via the AWS SDK for Python: [boto3](https://boto3.amazonaws.com/v1/documentation/api/latest/index.html).

#### Use different clients
The boto3 provides different clients for Amazon Bedrock to perform different actions. The actions for [`InvokeModel`](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_runtime_InvokeModel.html) and [`InvokeModelWithResponseStream`](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_runtime_InvokeModelWithResponseStream.html) are supported by Amazon Bedrock Runtime where as other operations, such as [ListFoundationModels](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_ListFoundationModels.html), are handled via [Amazon Bedrock client](https://docs.aws.amazon.com/bedrock/latest/APIReference/API_Operations_Amazon_Bedrock.html).


#### Use the default credential chain

If you are running this notebook from [Amazon Sagemaker Studio](https://aws.amazon.com/sagemaker/studio/) and your Sagemaker Studio [execution role](https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-roles.html) has permissions to access Bedrock you can just run the cells below as-is. This is also the case if you are running these notebooks from a computer whose default AWS credentials have access to Bedrock.



In [2]:
import json
import os
import sys

import boto3

boto3_bedrock = boto3.client('bedrock')

#### Validate the connection

We can check the client works by trying out the `list_foundation_models()` method, which will tell us all the models available for us to use 

In [3]:
[models['modelId'] for models in boto3_bedrock.list_foundation_models()['modelSummaries']]


['openai.gpt-6-astra',
 'nvidia.nemotron-nano-12b-v2',
 'qwen.qwen3-coder-next',
 'openai.gpt-5.6-terra',
 'anthropic.claude-haiku-4-5-20251001-v1:0',
 'anthropic.claude-opus-5-5',
 'moonshotai.kimi-k2.5',
 'openai.gpt-oss-120b-1:0',
 'stability.stable-creative-upscale-v1:0',
 'zai.glm-5.3',
 'qwen.qwen3-next-80b-a3b',
 'moonshotai.kimi-k3',
 'anthropic.claude-fable-5',
 'deepseek.v3.2',
 'anthropic.claude-haiku-5-5',
 'amazon.nova-2-multimodal-embeddings-v1:0',
 'twelvelabs.pegasus-1-5-v1:0',
 'nvidia.nemotron-nano-3-30b',
 'anthropic.claude-sonnet-4-6',
 'minimax.minimax-m2',
 'zai.glm-4.7-flash',
 'xai.grok-4.7',
 'mistral.voxtral-mini-3b-2507',
 'xai.grok-4.6',
 'amazon.nova-pro-v1:0',
 'amazon.nova-pro-v1:0:24k',
 'amazon.nova-pro-v1:0:300k',
 'stability.stable-image-remove-background-v1:0',
 'stability.stable-image-control-sketch-v1:0',
 'amazon.nova-2-lite-v1:0',
 'amazon.nova-2-lite-v1:0:256k',
 'stability.stable-conservative-upscale-v1:0',
 'minimax.minimax-m2.5',
 'google.gem

---

## `InvokeModel` body and output

The `invoke_model()` method of the Amazon Bedrock runtime client (`InvokeModel` API) will be the primary method we use for most of our Text Generation and Processing tasks - whichever model we're using.

Although the method is shared, the format of input and output varies depending on the foundation model used - as described below:

### Amazon Nova 2 Lite (Converse API)

Amazon Titan Text models reached end of life. Amazon's current text models are the Nova family, and the recommended way to call them is the [Converse API](https://docs.aws.amazon.com/bedrock/latest/userguide/conversation-inference-call.html): the same request and response shape for every model provider. Nova 2 Lite is called through its cross-Region inference profile `us.amazon.nova-2-lite-v1:0` ([model card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-amazon-nova-2-lite.html)).

#### Input (`converse` parameters)
```json
{
    "modelId": "us.amazon.nova-2-lite-v1:0",
    "messages": [{"role": "user", "content": [{"text": "<prompt>"}]}],
    "inferenceConfig": {"maxTokens": 512, "temperature": 0.1, "topP": 0.9}
}
```

#### Output
```json
{
    "output": {"message": {"role": "assistant", "content": [{"text": "<output>"}]}},
    "stopReason": "end_turn",
    "usage": {"inputTokens": 613, "outputTokens": 219, "totalTokens": 832},
    "metrics": {"latencyMs": 1234}
}
```

### Anthropic Claude (Messages API through InvokeModel)

Claude models use the Anthropic Messages API body. The example below is the one in the [Claude Haiku 5.5 model card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-anthropic-claude-haiku-5-5.html). Claude 4.7 and later do not accept `temperature`, `top_p` or `top_k` ([Anthropic docs](https://platform.claude.com/docs/en/models/haiku-5-5/overview)).

#### Input

```json
{
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 1024,
    "messages": [{"role": "user", "content": [{"type": "text", "text": "<prompt>"}]}]
}
```

Claude 5.5 has [adaptive thinking](https://platform.claude.com/docs/en/build-with-claude/thinking) on by default: the response can start with a `thinking` block (Invoke) or a `reasoningContent` block (Converse) before the `text` block, so the code below picks the text blocks explicitly instead of reading `content[0]`.

#### Output

```json
{
    "content": [
        {"type": "thinking", "thinking": "", "signature": "<signature>"},
        {"type": "text", "text": "<output>"}
    ],
    "stop_reason": "end_turn",
    "usage": {"input_tokens": 12, "output_tokens": 34}
}
```

### Stability AI Stable Diffusion 3.5 Large

Stable Diffusion XL reached end of life. The replacement is Stable Diffusion 3.5 Large (`stability.sd3-5-large-v1:0`), available in `us-west-2`. Request and response format from the [model parameters page](https://docs.aws.amazon.com/bedrock/latest/userguide/model-parameters-diffusion-3-5-large.html).

#### Input

```json
{
    "prompt": "this is where you place your input text",
    "mode": "text-to-image",
    "aspect_ratio": "1:1",
    "seed": 0,
    "output_format": "png"
}
```

#### Output

```json
{
    "seeds": [2130420379],
    "finish_reasons": [null],
    "images": ["<image in base64>"]
}
```

---

## Common inference parameter definitions

### Randomness and Diversity

Foundation models generally support the following parameters to control randomness and diversity in the 
response.

**Temperature** – Large language models use probability to construct the words in a sequence. For any 
given next word, there is a probability distribution of options for the next word in the sequence. When 
you set the temperature closer to zero, the model tends to select the higher-probability words. When 
you set the temperature further away from zero, the model may select a lower-probability word.

In technical terms, the temperature modulates the probability density function for the next tokens, 
implementing the temperature sampling technique. This parameter can deepen or flatten the density 
function curve. A lower value results in a steeper curve with more deterministic responses, and a higher 
value results in a flatter curve with more random responses.

**Top K** – Temperature defines the probability distribution of potential words, and Top K defines the cut 
off where the model no longer selects the words. For example, if K=50, the model selects from 50 of the 
most probable words that could be next in a given sequence. This reduces the probability that an unusual 
word gets selected next in a sequence.
In technical terms, Top K is the number of the highest-probability vocabulary tokens to keep for Top-
K-filtering - This limits the distribution of probable tokens, so the model chooses one of the highest-
probability tokens.

**Top P** – Top P defines a cut off based on the sum of probabilities of the potential choices. If you set Top 
P below 1.0, the model considers the most probable options and ignores less probable ones. Top P is 
similar to Top K, but instead of capping the number of choices, it caps choices based on the sum of their 
probabilities.
For the example prompt "I hear the hoof beats of ," you may want the model to provide "horses," 
"zebras" or "unicorns" as the next word. If you set the temperature to its maximum, without capping 
Top K or Top P, you increase the probability of getting unusual results such as "unicorns." If you set the 
temperature to 0, you increase the probability of "horses." If you set a high temperature and set Top K or 
Top P to the maximum, you increase the probability of "horses" or "zebras," and decrease the probability 
of "unicorns."

### Length

The following parameters control the length of the generated response.

**Response length** – Configures the minimum and maximum number of tokens to use in the generated 
response.

**Length penalty** – Length penalty optimizes the model to be more concise in its output by penalizing 
longer responses. Length penalty differs from response length as the response length is a hard cut off for 
the minimum or maximum response length.

In technical terms, the length penalty penalizes the model exponentially for lengthy responses. 0.0 
means no penalty. Set a value less than 0.0 for the model to generate longer sequences, or set a value 
greater than 0.0 for the model to produce shorter sequences.

### Repetitions

The following parameters help control repetition in the generated response.

**Repetition penalty (presence penalty)** – Prevents repetitions of the same words (tokens) in responses. 
1.0 means no penalty. Greater than 1.0 decreases repetition.

---

## Try out the models

With some theory out of the way, let's see the models in action! Run the cells below to see basic, synchronous example invocations for each model:

In [4]:
import boto3
import botocore
import json 

bedrock_runtime = boto3.client('bedrock-runtime')


### Amazon Nova 2 Lite (Converse API)

In [5]:
# If you'd like to try your own prompt, edit this parameter!
prompt_data = """Command: Write me a blog about making strong business decisions as a leader.

Blog:
"""


Next, we will construct the body with the `prompt_data` above, and add a optional parameters like `topP` and `temperature`:

In [6]:
try:
    modelId = "us.amazon.nova-2-lite-v1:0"

    response = bedrock_runtime.converse(
        modelId=modelId,
        messages=[{"role": "user", "content": [{"text": prompt_data}]}],
        inferenceConfig={"maxTokens": 512, "temperature": 0.2, "topP": 0.95},
    )

    print(response["output"]["message"]["content"][0]["text"])

except botocore.exceptions.ClientError as error:

    if error.response['Error']['Code'] == 'AccessDeniedException':
           print(f"\x1b[41m{error.response['Error']['Message']}\
                \nTo troubeshoot this issue please refer to the following resources.\
                 \nhttps://docs.aws.amazon.com/IAM/latest/UserGuide/troubleshoot_access-denied.html\
                 \nhttps://docs.aws.amazon.com/bedrock/latest/userguide/security-iam.html\x1b[0m\n")

    else:
        raise error

## **Making Strong Business Decisions: A Leader’s Guide to Strategic Success**

In the fast-paced and ever-evolving world of business, one constant remains true: **strong decision-making is the hallmark of effective leadership**. Whether you're steering a startup through its early days or guiding a multinational corporation through market disruptions, your ability to make informed, confident decisions will determine your organization’s success.

But what does it mean to make *strong* business decisions? And how can leaders cultivate this critical skill? Let’s explore the mindset, processes, and habits that define exceptional decision-making in leadership.

---

### **1. Embrace Responsibility, Not Fear**

One of the first steps in becoming a strong decision-maker is to **accept that decisions are part of the job**. Leaders often struggle with the fear of making the wrong choice—but waiting for perfect clarity is a recipe for paralysis.

> **Strong leaders don’t fear decisions; they fea

### Stability AI Stable Diffusion 3.5 Large

In [7]:
prompt_data = "a landscape with trees"
body = json.dumps({
    "prompt": prompt_data,
    "mode": "text-to-image",
    "aspect_ratio": "1:1",
    "seed": 20,
    "output_format": "png",
})
modelId = "stability.sd3-5-large-v1:0"
accept = "application/json"
contentType = "application/json"

# Stable Diffusion 3.5 Large is served from us-west-2
bedrock_runtime_west = boto3.client("bedrock-runtime", region_name="us-west-2")

try:

    response = bedrock_runtime_west.invoke_model(
        body=body, modelId=modelId, accept=accept, contentType=contentType
    )
    response_body = json.loads(response.get("body").read())

    print(f'finish_reasons: {response_body["finish_reasons"]}  seeds: {response_body["seeds"]}')
    print(f'{response_body["images"][0][0:80]}...')

except botocore.exceptions.ClientError as error:

    if error.response['Error']['Code'] == 'AccessDeniedException':
           print(f"\x1b[41m{error.response['Error']['Message']}\
                \nTo troubeshoot this issue please refer to the following resources.\
                 \nhttps://docs.aws.amazon.com/IAM/latest/UserGuide/troubleshoot_access-denied.html\
                 \nhttps://docs.aws.amazon.com/bedrock/latest/userguide/security-iam.html\x1b[0m\n")

    else:
        raise error

finish_reasons: [None]  seeds: [20]
iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAABWZkRUNSJJPjAOWrYpkAIwZKSURBVHgB...


**Note:** The output is a [base64 encoded](https://docs.python.org/3/library/base64.html) string of the image data. You can use any image processing library (such as [Pillow](https://pillow.readthedocs.io/en/stable/)) to decode the image as in the example below:

```python
import base64
import io
from PIL import Image

base_64_img_str = response_body.get("artifacts")[0].get("base64")
image = Image.open(io.BytesIO(base64.decodebytes(bytes(base_64_img_str, "utf-8"))))
```

In [8]:
%pip install pillow
import base64
import io
from PIL import Image

base_64_img_str = response_body["images"][0]
image = Image.open(io.BytesIO(base64.decodebytes(bytes(base_64_img_str, "utf-8"))))
image

## Generate streaming output

For large language models, it can take noticeable time to generate long output sequences. Rather than waiting for the entire response to be available, latency-sensitive applications may like to **stream** the response to users.

Run the code below to see how you can achieve this with Bedrock's `invoke_model_with_response_stream()` method - returning the response body in separate chunks.

In [9]:
from IPython.display import clear_output, display, display_markdown, Markdown
prompt_data = """Command: Write me a blog about making strong business decisions as a leader.

Blog:
"""

modelId = "us.amazon.nova-2-lite-v1:0"  # (Change this to try different models: the Converse API keeps the same syntax)

try:

    response = bedrock_runtime.converse_stream(
        modelId=modelId,
        messages=[{"role": "user", "content": [{"text": prompt_data}]}],
        inferenceConfig={"maxTokens": 512},
    )

    for event in response["stream"]:
        if "contentBlockDelta" in event:
            print(event["contentBlockDelta"]["delta"]["text"], end="")

except botocore.exceptions.ClientError as error:

    if error.response['Error']['Code'] == 'AccessDeniedException':
           print(f"\x1b[41m{error.response['Error']['Message']}\
                \nTo troubeshoot this issue please refer to the following resources.\
                 \nhttps://docs.aws.amazon.com/IAM/latest/UserGuide/troubleshoot_access-denied.html\
                 \nhttps://docs.aws.amazon.com/bedrock/latest/userguide/security-iam.html\x1b[0m\n")

    else:
        raise error

## **Making

 Strong Business Decisions: A Leader’s Guide to Strategic Success**

As a leader, one of your most critical responsibilities is making decisions—decisions that can shape the trajectory of your organization, impact your team, and determine long-term success. In today

’s fast-paced and ever-changing business landscape, the ability to make strong, informed, and timely decisions is a hallmark of effective leadership.

But what does it mean to make a *strong* business decision? It goes beyond gut feeling or quick

 choices under pressure. It involves a blend of strategy, empathy, data, and vision. In this blog, we’ll explore the key elements of strong decision-making and how leaders can cultivate this essential skill.

---

###

 **1. Understand the Context Before You Decide**

Strong decisions are never made in a vacuum. Before taking action, take the time to understand the full context:

- **Market Conditions:** What’s happening in your industry? Are there emerging

 trends, disruptions, or threats?
- **Organizational Goals:** How does this decision align with your company’s mission, vision, and strategic objectives?
- **Team Dynamics:** Who will be affected? How will the decision impact morale, resources, and workflows

... (2 outputs left out)

 noise to find what truly matters.

**Best practices for data-driven decision-making:**

- **Define Clear Objectives:** Know what questions you’re trying to answer.
- **Use Reliable Sources:** Rely on verified internal reports, market research, and analytics

 tools.
- **Balance Quantitative and Qualitative Insights:** Numbers tell part of the story; customer feedback, team input, and leadership intuition complete it.

However, beware of *analysis paralysis*—the tendency

 to delay action while seeking perfect information. At some point, you must move from evaluation to execution.

---

### **3. Consider the Long-Term Impact**

Strong leaders think beyond the next quarter or

 the next performance review. They consider the long-term consequences of their decisions.

Ask yourself:

- **Will this decision strengthen our culture or erode it?**
- **Does it position us for future growth, or could it create technical or financial debt?**
-

### Anthropic Claude (messages API)

Streaming events follow the [Anthropic streaming format](https://platform.claude.com/docs/en/build-with-claude/streaming): text arrives in `content_block_delta` events whose `delta.type` is `text_delta`; thinking deltas are skipped. These demos ask for short answers with a small token budget, so [adaptive thinking](https://platform.claude.com/docs/en/build-with-claude/thinking) is turned off explicitly with `"thinking": {"type": "disabled"}`; otherwise the model can spend the whole `max_tokens` budget thinking and return no text.

In [10]:
# If you'd like to try your own prompt, edit this parameter!
prompt_data = """Human: Write me 500 word paragraph about making strong business decisions as a leader.

Assistant:
"""


In [11]:
messages_API_body = {
    "anthropic_version": "bedrock-2023-05-31", 
    "max_tokens": int(500/0.75),
    "thinking": {"type": "disabled"},
    "messages": [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt_data
                }
            ]
        }
    ]
}

In [12]:
from IPython.display import clear_output, display, display_markdown, Markdown

body = json.dumps(messages_API_body)
modelId = "global.anthropic.claude-haiku-5-5"  # (Change this to try different model versions)
accept = "application/json"
contentType = "application/json"

try:

    response = bedrock_runtime.invoke_model_with_response_stream(
        body=body, modelId=modelId, accept=accept, contentType=contentType
    )
    
    stream = response.get('body')

    if stream:
        for event in stream:
            chunk = event.get('chunk')
            if chunk:
                chunk_obj = json.loads(chunk.get('bytes').decode())
                if chunk_obj.get('type') == 'content_block_delta':
                    delta_obj = chunk_obj.get('delta', {})
                    if delta_obj.get('type') == 'text_delta':
                        print(delta_obj.get('text', ''), end='')

except botocore.exceptions.ClientError as error:

    if error.response['Error']['Code'] == 'AccessDeniedException':
           print(f"\x1b[41m{error.response['Error']['Message']}\
                \nTo troubeshoot this issue please refer to the following resources.\
                 \nhttps://docs.aws.amazon.com/IAM/latest/UserGuide/troubleshoot_access-denied.html\
                 \nhttps://docs.aws.amazon.com/bedrock/latest/userguide/security-iam.html\x1b[0m\n")

    else:
        raise error


Making strong business decisions as a leader begins with clarity about purpose. Before weighing options, a

 leader must understand what the organization is trying to achieve and what values should guide its choices. Decisions made without a clear sense of direction tend to drift, pulling resources toward wh

atever problem is loudest rather than whatever matters most. Anchoring each decision to the company's mission and long-term goals creates a consistent filter that makes trade-offs easier to ev

aluate and easier to explain to the people who must carry them out.

Equally important is the quality of information a leader relies upon. Strong decisions are rarely the product of intuition alone, though intuition has

 its place when it is informed by experience. Effective leaders seek out diverse perspectives, particularly from those closest to customers and operations, who often see problems that executives mi

ss. They distinguish between data that is merely available and data that is genuinely relevant, and they remain alert to the biases that can distort their own judgment, such as over

... (3 outputs left out)

 identifying the range of plausible outcomes, estimating the costs of being wrong, and designing decisions that can be reversed or adjusted when new evidence emerges. Staging invest

ments, running pilot programs, and setting clear checkpoints for review allow an organization to learn quickly and limit exposure. A leader who treats decisions as experiments rather than irrevocable commitments buil

ds an organization that adapts rather than one that doubles down on errors.

Finally, the most consequential decisions depend on the people who implement them. A choice that is techn

ically sound will fail if those responsible for executing it do not understand it, believe in it, or feel empowered to adjust it. Strong leaders communicate the reasoning behind their choices

## Generate embeddings

Use text embeddings to convert text into meaningful vector representations. You input a body of text 
and the output is a (1 x n) vector. You can use embedding vectors for a wide variety of applications. 
Bedrock currently offers Titan Embeddings for text embedding that supports text similarity (finding the 
semantic similarity between bodies of text) and text retrieval (such as search).

At the time of writing you can use `amazon.titan-embed-text-v1` as embedding model via the API. The input text size is 8192 tokens and the output vector length is 1536.

To use a text embeddings model, use the InvokeModel API operation or the Python SDK.
Use InvokeModel to retrieve the vector representation of the input text from the specified model.



#### Input

```json
{
    "inputText": "<text>"
}
```

#### Output

```json
{
    "embedding": []
}
```


Let's see how to generate embeddings of some text:

In [13]:
prompt_data = "Amazon Bedrock supports foundation models from industry-leading providers such as \
AI21 Labs, Anthropic, Stability AI, and Amazon. Choose the model that is best suited to achieving \
your unique goals."


In [14]:
body = json.dumps({"inputText": prompt_data})
modelId = "amazon.titan-embed-g1-text-02"  # (Change this to try different embedding models)
accept = "application/json"
contentType = "application/json"

try:

    response = bedrock_runtime.invoke_model(
        body=body, modelId=modelId, accept=accept, contentType=contentType
    )
    response_body = json.loads(response.get("body").read())

    embedding = response_body.get("embedding")
    print(f"The embedding vector has {len(embedding)} values\n{embedding[0:3]+['...']+embedding[-3:]}")

except botocore.exceptions.ClientError as error:

    if error.response['Error']['Code'] == 'AccessDeniedException':
           print(f"\x1b[41m{error.response['Error']['Message']}\
                \nTo troubeshoot this issue please refer to the following resources.\
                 \nhttps://docs.aws.amazon.com/IAM/latest/UserGuide/troubleshoot_access-denied.html\
                 \nhttps://docs.aws.amazon.com/bedrock/latest/userguide/security-iam.html\x1b[0m\n")

    else:
        raise error


The embedding vector has 1536 values
[0.166015625, 0.236328125, 0.703125, '...', 0.26953125, -0.609375, -0.55078125]


## Next steps

In this notebook we showed some basic examples of invoking Amazon Bedrock models using the AWS Python SDK. You're now ready to explore the other labs to dive deeper on different use-cases and patterns.